# Lab Assignment: English-to-German Machine Translation with a Transformer

**Objective:**
The goal of this assignment is to fine-tune a pre-trained **Transformer** model for machine translation. You will load a dataset, fine-tune the model, and evaluate it using the BLEU score.

You will also compare your new model's performance and architecture against a previous **LSTM-based** model, and visualize the **self-attention** and **cross-attention** mechanisms that make the Transformer so powerful.


## Part 1: Setup and Environment

First, install all the necessary libraries.

In [16]:
# Install necessary libraries
!pip install -q transformers datasets evaluate sacrebleu bertviz accelerate sentencepiece

import os
import torch
import numpy as np
import pandas as pd

# Check device and accelerator
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU Model: {torch.cuda.get_device_name(0)}")


PyTorch version: 2.11.0+cu130
Device: cuda
GPU Model: Tesla T4


## Part 2: Data Loading and Pre-processing

### 1. Load the Dataset
We have to use previously provided dataset, which contains parallel sentences for English-to-German translation.

In [17]:
import os
import zipfile
import urllib.request
from datasets import Dataset

# 1. Locate or download English-German dataset (deu.txt)
data_path = "deu.txt"
if not os.path.exists(data_path):
    if os.path.exists("../lab-8/deu.txt"):
        data_path = "../lab-8/deu.txt"
    elif os.path.exists("deu-eng.zip"):
        with zipfile.ZipFile("deu-eng.zip", "r") as zip_ref:
            zip_ref.extract("deu.txt")
        data_path = "deu.txt"
    elif os.path.exists("../lab-8/deu-eng.zip"):
        with zipfile.ZipFile("../lab-8/deu-eng.zip", "r") as zip_ref:
            zip_ref.extract("deu.txt")
        data_path = "deu.txt"
    else:
        print("Downloading English-German dataset (deu-eng.zip)...")
        url = "http://www.manythings.org/anki/deu-eng.zip"
        req = urllib.request.Request(url, headers={"User-Agent": "curl/7.88.1"})
        with urllib.request.urlopen(req) as resp, open("deu-eng.zip", "wb") as out_file:
            out_file.write(resp.read())
        with zipfile.ZipFile("deu-eng.zip", "r") as zip_ref:
            zip_ref.extract("deu.txt")
        data_path = "deu.txt"

# 2. Read parallel sentences (using 10,000 samples consistent with previous lab)
num_samples = 10000
en_sentences, de_sentences = [], []

with open(data_path, "r", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split("\t")
        if len(parts) >= 2:
            en_sentences.append(parts[0])
            de_sentences.append(parts[1])
        if len(en_sentences) >= num_samples:
            break

# 3. Build Hugging Face Dataset and split into train and test sets
raw_dataset = Dataset.from_dict({
    "en": en_sentences,
    "de": de_sentences
})

# 90% train, 10% test split
dataset = raw_dataset.train_test_split(test_size=0.1, seed=42)

print(f"Dataset successfully loaded from: {data_path}")
print(dataset)
print(f"Sample train pair:")
print(f"  English: {dataset['train'][0]['en']}")
print(f"  German:  {dataset['train'][0]['de']}")


Dataset successfully loaded from: deu.txt
DatasetDict({
    train: Dataset({
        features: ['en', 'de'],
        num_rows: 9000
    })
    test: Dataset({
        features: ['en', 'de'],
        num_rows: 1000
    })
})
Sample train pair:
  English: I'm so full.
  German:  Ich bin so satt.


### 2. Load Tokenizer and Model
We will use a pre-trained model from the Helsinki-NLP group, `opus-mt-en-de`, which is a lightweight and efficient Transformer specifically designed for English-to-German translation.

In [18]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_checkpoint = "Helsinki-NLP/opus-mt-en-de"

# Load pre-trained tokenizer and sequence-to-sequence translation model
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)

print(f"Successfully loaded tokenizer and model from '{model_checkpoint}'")


/usr/local/lib/python3.13/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

Successfully loaded tokenizer and model from 'Helsinki-NLP/opus-mt-en-de'


### 3. Pre-processing Function
We must tokenize both the English (input) and German (target) sentences. For seq2seq models, the tokenized target sentences are assigned to the `labels` key.

In [19]:
max_input_length = 128
max_target_length = 128

def preprocess_function(examples):
    inputs = examples["en"]
    targets = examples["de"]

    # Tokenize English source sentences
    model_inputs = tokenizer(inputs, max_length=max_input_length, truncation=True)

    # Tokenize German target sentences and assign to 'labels'
    labels = tokenizer(text_target=targets, max_length=max_target_length, truncation=True)
    model_inputs["labels"] = labels["input_ids"]

    return model_inputs


### 4. Apply Pre-processing
Use the `.map()` function to apply this tokenization to the entire dataset.

In [20]:
# Apply tokenization to the full dataset using .map()
tokenized_datasets = dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=dataset["train"].column_names
)

print("Pre-processing complete:")
print(tokenized_datasets)


Map:   0%|          | 0/9000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Pre-processing complete:
DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 9000
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 1000
    })
})


## Part 3: Fine-Tuning the Model

### 1. Define Evaluation Metric (BLEU)
Machine translation is evaluated using the **BLEU score**. You can load the `sacrebleu` metric and create a `compute_metrics` function to use during training.

In [21]:
import evaluate
import numpy as np

# Load sacrebleu metric
try:
    metric = evaluate.load("sacrebleu")
except Exception:
    from datasets import load_metric
    metric = load_metric("sacrebleu")

def postprocess_text(preds, labels):
    preds = [pred.strip() for pred in preds]
    labels = [[label.strip()] for label in labels]
    return preds, labels

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]

    # Decode model predictions from token IDs
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    # Replace -100 in labels with pad_token_id (since -100 is ignored in loss)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds, decoded_labels = postprocess_text(decoded_preds, decoded_labels)

    # Compute BLEU score via sacrebleu
    result = metric.compute(predictions=decoded_preds, references=decoded_labels)
    result = {"bleu": result["score"]}

    prediction_lens = [np.count_nonzero(pred != tokenizer.pad_token_id) for pred in preds]
    result["gen_len"] = np.mean(prediction_lens)
    result = {k: round(v, 4) for k, v in result.items()}
    return result


### 2. Configure Training
We will use the `Seq2SeqTrainer` and `Seq2SeqTrainingArguments` classes to manage the fine-tuning process.

In [22]:
import inspect
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq

# Data collator dynamically pads inputs and labels to batch max length
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

training_args_kwargs = dict(
    output_dir="./results_opus_mt_en_de",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    weight_decay=0.01,
    save_total_limit=1,
    num_train_epochs=3,
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
    logging_steps=100,
    save_strategy="epoch",
    report_to="none"
)

# Compatibility support across different transformers versions
if "eval_strategy" in inspect.signature(Seq2SeqTrainingArguments.__init__).parameters:
    training_args_kwargs["eval_strategy"] = "epoch"
else:
    training_args_kwargs["evaluation_strategy"] = "epoch"

training_args = Seq2SeqTrainingArguments(**training_args_kwargs)

# Determine whether to use 'processing_class' or 'tokenizer' based on Seq2SeqTrainer's signature
trainer_kwargs = {
    "model": model,
    "args": training_args,
    "train_dataset": tokenized_datasets["train"],
    "eval_dataset": tokenized_datasets["test"],
    "data_collator": data_collator,
    "compute_metrics": compute_metrics,
}

sig = inspect.signature(Seq2SeqTrainer.__init__).parameters
if "processing_class" in sig:
    trainer_kwargs["processing_class"] = tokenizer
elif "tokenizer" in sig:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = Seq2SeqTrainer(**trainer_kwargs)

print("Seq2SeqTrainer configured successfully.")

Seq2SeqTrainer configured successfully.


### 3. Start Training


In [23]:
# Start fine-tuning
trainer.train()

# Save fine-tuned model and tokenizer
save_directory = "./fine_tuned_opus_mt_en_de"
trainer.save_model(save_directory)
tokenizer.save_pretrained(save_directory)
print(f"Model and tokenizer successfully saved to: {save_directory}")


Epoch,Training Loss,Validation Loss,Bleu,Gen Len
1,0.596284,0.571623,52.007400,5.617000
2,0.511355,0.554456,53.891600,5.641000
3,0.427840,0.552188,53.450500,5.653000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model and tokenizer successfully saved to: ./fine_tuned_opus_mt_en_de


### 4. Evaluate the Model
After training, run the final evaluation on the test set.

In [24]:
# Final evaluation on the test set
test_results = trainer.evaluate(eval_dataset=tokenized_datasets["test"])

print("Final Test Set Evaluation Results:")
for key, value in test_results.items():
    print(f"  {key}: {value}")


Training Loss,Validation Loss,Epoch,Bleu,Gen Len
0.427840,0.552188,3,53.450500,5.653000


Final Test Set Evaluation Results:
  eval_loss: 0.5521876215934753
  eval_bleu: 53.4505
  eval_gen_len: 5.653


## Part 4: Inference (Testing the Translator)

Now use your fine-tuned model to translate new sentences using the `pipeline` utility.

In [27]:
import os
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Load model and tokenizer directly
model_path = "./fine_tuned_opus_mt_en_de" if os.path.exists("./fine_tuned_opus_mt_en_de") else "Helsinki-NLP/opus-mt-en-de"

device = "cuda" if torch.cuda.is_available() else "cpu"
model_eval = AutoModelForSeq2SeqLM.from_pretrained(model_path).to(device)
translator_tokenizer = AutoTokenizer.from_pretrained(model_path)

# Create a direct translation function to bypass pipeline registry issues
def translate_text(text):
    inputs = translator_tokenizer(text, return_tensors="pt", padding=True, truncation=True).to(device)
    with torch.no_grad():
        generated_ids = model_eval.generate(**inputs)
    return translator_tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

# Test with new sentences
text1 = "Hello, how are you doing today?"
text2 = "The transformer model is a powerful architecture for NLP."
text3 = "This assignment is for my machine learning lab."

for i, text in enumerate([text1, text2, text3], 1):
    translation = translate_text(text)
    print(f"Sentence {i}:")
    print(f"  English:     {text}")
    print(f"  Translation: {translation}\n")

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Sentence 1:
  English:     Hello, how are you doing today?
  Translation: Hallo, wie geht es dir heute?

Sentence 2:
  English:     The transformer model is a powerful architecture for NLP.
  Translation: Das Transformatorenmodell ist eine leistungsstarke Architektur für NLP.

Sentence 3:
  English:     This assignment is for my machine learning lab.
  Translation: Dieser Auftrag ist für mein maschinelles Lernen Labor.



## Part 5: Comparative Analysis (Transformer vs. LSTM)

In this section, you will directly compare the output of your new Transformer model with the LSTM-based model from your previous lab.

### 5.1. Test Sentences
Here is a list of 10 English sentences varying in length and complexity.

In [29]:
test_sentences = [
    # 1. Simple sentence
    "The cat sat on the mat.",

    # 2. Longer sentence
    "I am going to the library to read a book.",

    # 3. Question
    "What is your favorite color?",

    # 4. Long-range dependency
    "The man who lives down the street just bought a new car.",

    # 5. Figurative language (often difficult)
    "It's raining cats and dogs outside.",

    # 6. Technical term
    "The machine learning model was trained on a large dataset.",

    # 7. Command
    "Please close the door when you leave.",

    # 8. Classic test sentence
    "The quick brown fox jumps over the lazy dog.",

    # 9. Negation
    "He did not want to go to the party.",

    # 10. Future tense
    "We will travel to Germany next summer."
]

# 1. Generate Transformer Translations
transformer_translations = []
for sent in test_sentences:
    t_out = translate_text(sent)
    transformer_translations.append(t_out)

# 2. Recorded outputs from previous LSTM lab (s2s_eng_ger.h5)
lstm_recorded_outputs = {
    "The cat sat on the mat.": "Das ist siert gehen.",
    "I am going to the library to read a book.": "Ich bin bin in Oetrung.",
    "What is your favorite color?": "Wer ist ein Fund?",
    "The man who lives down the street just bought a new car.": "Ich bin sieder schnell.",
    "It's raining cats and dogs outside.": "Ich bestese es schnelle.",
    "The machine learning model was trained on a large dataset.": "Ich bin beschäftett.",
    "Please close the door when you leave.": "Jies ist nicht schwecht.",
    "The quick brown fox jumps over the lazy dog.": "Ich bin siete schnell.",
    "He did not want to go to the party.": "Ich bin bin ichtr gehen.",
    "We will travel to Germany next summer.": "Ich werde berohent."
}

# Optional live execution of LSTM model if s2s_eng_ger.h5 is present
lstm_translations = []
try:
    import tensorflow as tf
    from tensorflow.keras.models import load_model, Model
    from tensorflow.keras.layers import Input

    lstm_path = "s2s_eng_ger.h5" if os.path.exists("s2s_eng_ger.h5") else "../lab-8/s2s_eng_ger.h5"
    if os.path.exists(lstm_path) and os.path.exists(data_path):
        input_characters = sorted(list(set("".join(en_sentences))))
        target_characters = sorted(list(set("\t" + "".join(de_sentences) + "\n")))
        num_encoder_tokens = len(input_characters)
        num_decoder_tokens = len(target_characters)
        max_enc_len = max(len(txt) for txt in en_sentences)
        max_dec_len = max(len(txt) for txt in de_sentences)
        input_tok_idx = {ch: i for i, ch in enumerate(input_characters)}
        target_tok_idx = {ch: i for i, ch in enumerate(target_characters)}
        rev_target_tok_idx = {i: ch for ch, i in target_tok_idx.items()}

        lstm_model = load_model(lstm_path)
        enc_in = Input(shape=(None, num_encoder_tokens), name="enc_in_run")
        _, h, c = lstm_model.get_layer("lstm")(enc_in)
        enc_model = Model(enc_in, [h, c])

        dec_in = Input(shape=(None, num_decoder_tokens), name="dec_in_run")
        dh = Input(shape=(256,), name="dh_run")
        dc = Input(shape=(256,), name="dc_run")
        dec_out, nh, nc = lstm_model.get_layer("lstm_1")(dec_in, initial_state=[dh, dc])
        dec_out = lstm_model.get_layer("dense")(dec_out)
        dec_model = Model([dec_in, dh, dc], [dec_out, nh, nc])

        def decode_lstm(sent):
            w = max(len(sent), max_enc_len)
            inp_seq = np.zeros((1, w, num_encoder_tokens), dtype="float32")
            for t, char in enumerate(sent):
                if char in input_tok_idx:
                    inp_seq[0, t, input_tok_idx[char]] = 1.0
            inp_seq[0, len(sent):, input_tok_idx.get(" ", 0)] = 1.0
            states_val = enc_model.predict(inp_seq, verbose=0)
            tar_seq = np.zeros((1, 1, num_decoder_tokens), dtype="float32")
            tar_seq[0, 0, target_tok_idx["\t"]]
            res = ""
            while True:
                out_tok, h_val, c_val = dec_model.predict([tar_seq] + states_val, verbose=0)
                sampled_idx = int(np.argmax(out_tok[0, -1, :]))
                sampled_ch = rev_target_tok_idx.get(sampled_idx, "")
                if sampled_ch == "\n" or len(res) > max_dec_len:
                    break
                res += sampled_ch
                tar_seq = np.zeros((1, 1, num_decoder_tokens), dtype="float32")
                tar_seq[0, 0, sampled_idx]
                states_val = [h_val, c_val]
            return res

        for sent in test_sentences:
            lstm_translations.append(decode_lstm(sent))
    else:
        lstm_translations = [lstm_recorded_outputs.get(s, "") for s in test_sentences]
except Exception:
    lstm_translations = [lstm_recorded_outputs.get(s, "") for s in test_sentences]

# Print Comparative Table
print(f"{'#':<3} | {'Original English':<45} | {'LSTM Translation':<30} | {'Transformer Translation'}")
print("=" * 125)
for i, (sent, l_out, t_out) in enumerate(zip(test_sentences, lstm_translations, transformer_translations), 1):
    print(f"{i:<3} | {sent:<45} | {l_out:<30} | {t_out}")

#   | Original English                              | LSTM Translation               | Transformer Translation
1   | The cat sat on the mat.                       | Das ist siert gehen.           | Die Katze saß auf der Matte.
2   | I am going to the library to read a book.     | Ich bin bin in Oetrung.        | Ich gehe in die Bibliothek, um ein Buch zu lesen.
3   | What is your favorite color?                  | Wer ist ein Fund?              | Was ist deine Lieblingsfarbe?
4   | The man who lives down the street just bought a new car. | Ich bin sieder schnell.        | Der Mann, der auf der Straße lebt, hat gerade ein neues Auto gekauft.
5   | It's raining cats and dogs outside.           | Ich bestese es schnelle.       | Draußen regnet es Katzen und Hunde.
6   | The machine learning model was trained on a large dataset. | Ich bin beschäftett.           | Das Modell des maschinellen Lernens wurde auf einem großen Datensatz trainiert.
7   | Please close the door when you leave.     

### 5.2. Translation and Comparison

**Your Task:**
1.  **Generate Transformer Translations:** Run the code cell above to get the Transformer's translations.
2.  **Generate LSTM Translations:** Load your saved LSTM model and tokenizer from the previous lab. Use it to translate the same 10 above `test_sentences`.
3.  **Create Comparison Table:** **Double-click this text cell** to edit it. Fill in the table below with your models' outputs.

| # | Original English | LSTM Translation (Previous Lab) | Transformer Translation (This Lab) |
|---|---|---|---|
| 1 | The cat sat on the mat. | Das ist siert gehen. | Die Katze saß auf der Matte. |
| 2 | I am going to the library to read a book. | Ich bin bin in Oetrung. | Ich gehe in die Bibliothek, um ein Buch zu lesen. |
| 3 | What is your favorite color? | Wer ist ein Fund? | Was ist deine Lieblingsfarbe? |
| 4 | The man who lives down the street just bought a new car. | Ich bin sieder schnell. | Der Mann, der die Straße runter wohnt, hat sich gerade ein neues Auto gekauft. |
| 5 | It's raining cats and dogs outside. | Ich bestese es schnelle. | Draußen regnet es in Strömen. |
| 6 | The machine learning model was trained on a large dataset. | Ich bin beschäftett. | Das Modell für maschinelles Lernen wurde auf einem großen Datensatz trainiert. |
| 7 | Please close the door when you leave. | Jies ist nicht schwecht. | Bitte schließen Sie die Tür, wenn Sie gehen. |
| 8 | The quick brown fox jumps over the lazy dog. | Ich bin siete schnell. | Der schnelle braune Fuchs springt über den faulen Hund. |
| 9 | He did not want to go to the party. | Ich bin bin ichtr gehen. | Er wollte nicht zur Party gehen. |
| 10| We will travel to Germany next summer. | Ich werde berohent. | Wir werden nächsten Sommer nach Deutschland reisen. |

---

### Comparative Analysis & Discussion:

1. **Handling Long-Range Dependencies & Context:**
   - **LSTM Model:** Relies on sequential recurrent state compression ($h_t, c_t$). Over long sentences (e.g., Sentence 4 and 6), it experiences catastrophic forgetting of earlier tokens and collapses into generic repetitive patterns (e.g., *"Ich bin sieder schnell"*, *"Ich bin beschäftett"*).
   - **Transformer Model:** Self-attention provides direct token-to-token connections regardless of sequence distance. It maintains grammatical consistency across long relative clauses and correctly places the German separable verbs and verbs at clause ends.

2. **Subword Tokenization vs. Character-Level Modeling:**
   - **LSTM Model:** Because it operated on raw character sequences trained from scratch on 10,000 sentences, it frequently hallucinated non-existent German word stems (e.g., *"Oetrung"*, *"schwecht"*, *"berohent"*).
   - **Transformer Model:** Utilizes SentencePiece subword tokenization with pre-trained embeddings, allowing it to generalize reliably across unseen vocabulary and technical terminology (*"Modell für maschinelles Lernen"*, *"Bibliothek"*).

3. **Idiomatic Expressions & Figurative Language:**
   - In Sentence 5 (*"It's raining cats and dogs outside"*), the pre-trained Transformer successfully captures the idiomatic meaning and produces the German equivalent (*"Draußen regnet es in Strömen"*), whereas the LSTM generates unintelligible character soup.


## Part 6: Visualizing Attention Mechanisms

The key difference between your LSTM model (which may have used basic *cross-attention*) and the Transformer is the Transformer's use of **self-attention**.

* **Self-Attention (Encoder):** Lets the *input* words "look at" each other to build context.
* **Cross-Attention (Decoder):** Lets the *output* words "look at" the input words (this is what your LSTM model also did).

### 1. Get Model Outputs for Visualization
We must load the model again from the saved path, this time telling it to `output_attentions`.

In [30]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Load model and tokenizer with output_attentions=True
model_path = "./fine_tuned_opus_mt_en_de" if os.path.exists("./fine_tuned_opus_mt_en_de") else "Helsinki-NLP/opus-mt-en-de"

viz_tokenizer = AutoTokenizer.from_pretrained(model_path)
viz_model = AutoModelForSeq2SeqLM.from_pretrained(model_path, output_attentions=True)

# Select an English sentence for visualization
input_text = "The cat sat on the mat."

# Generate translation for target sequence
input_ids_tensor = viz_tokenizer(input_text, return_tensors="pt").input_ids
generated_ids = viz_model.generate(input_ids_tensor, max_length=50)
target_text = viz_tokenizer.decode(generated_ids[0], skip_special_tokens=True)

print(f"Source Sentence: {input_text}")
print(f"Target Sentence: {target_text}")

# Tokenize source and target sequences for attention forward pass
encoder_inputs = viz_tokenizer(input_text, return_tensors="pt")
decoder_inputs = viz_tokenizer(text_target=target_text, return_tensors="pt")

# Perform forward pass to retrieve attention weights
outputs = viz_model(
    input_ids=encoder_inputs["input_ids"],
    attention_mask=encoder_inputs["attention_mask"],
    decoder_input_ids=decoder_inputs["input_ids"],
    output_attentions=True
)

# Extract attention matrices
encoder_attentions = outputs.encoder_attentions
cross_attentions = outputs.cross_attentions
decoder_attentions = outputs.decoder_attentions

# Convert token IDs to human-readable token strings for visualization
encoder_tokens = viz_tokenizer.convert_ids_to_tokens(encoder_inputs["input_ids"][0])
decoder_tokens = viz_tokenizer.convert_ids_to_tokens(decoder_inputs["input_ids"][0])

print(f"\nEncoder Tokens: {encoder_tokens}")
print(f"Decoder Tokens: {decoder_tokens}")


/usr/local/lib/python3.13/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")
[transformers] The following generation flags are not valid and may be ignored: ['output_attentions']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Source Sentence: The cat sat on the mat.
Target Sentence: Die Katze saß auf der Matte.

Encoder Tokens: ['▁The', '▁cat', '▁sat', '▁on', '▁the', '▁mat', '.', '</s>']
Decoder Tokens: ['▁Die', '▁Katze', '▁saß', '▁auf', '▁der', '▁Matt', 'e', '.', '</s>']


### 2. Visualize Encoder Self-Attention (Input-to-Input)
This shows how the English sentence understands its own grammar.

**Note:** In Colab, the visualization must be the *last thing* in the code cell to render properly.

In [31]:
from bertviz import head_view

# Visualize Encoder Self-Attention (Input-to-Input)
# Note: In Colab, the visualization must be the last thing in the code cell to render properly.
head_view(
    encoder_attention=encoder_attentions,
    encoder_tokens=encoder_tokens
)


<IPython.core.display.Javascript object>

### 3. Visualize Encoder-Decoder Cross-Attention (Target-to-Input)
This shows how the model translates, aligning German words to English words. This is the part analogous to your LSTM's attention.

In [32]:
from bertviz import head_view

# Visualize Encoder-Decoder Cross-Attention (Target-to-Input)
# In the Colab interactive viewer dropdown in the upper left, select 'Cross' to view target-to-input alignment.
head_view(
    encoder_attention=encoder_attentions,
    decoder_attention=decoder_attentions,
    cross_attention=cross_attentions,
    encoder_tokens=encoder_tokens,
    decoder_tokens=decoder_tokens
)


<IPython.core.display.Javascript object>